# Pretrained WideResNet50-2 Backbone Notebook

This notebook evaluates a frozen ImageNet-pretrained `WideResNet50-2` backbone on the shared `64x64` 5% test-defect split without using an autoencoder or PatchCore.

In [ ]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation import summarize_threshold_metrics, sweep_threshold_metrics

from torchvision.models import Wide_ResNet50_2_Weights, wide_resnet50_2

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "training" / "train_wideresnet50_backbone.toml"
config = load_toml(CONFIG_PATH)
PCA_MAX_POINTS_PER_GROUP = 500

# Optional quick smoke-run overrides:
# config["data"]["metadata_csv"] = "data/processed/x64/wm811k/metadata_dev.csv"
# config["run"]["output_dir"] = "artifacts/x64/wideresnet50_embedding_dev"

config

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)

set_seed(int(config["run"]["seed"]))

device = resolve_device(config["training"].get("device", "auto"))
print("Resolved device:", device)
print("CUDA available:", torch.cuda.is_available())
device


In [ ]:
image_size = int(config["data"].get("image_size", 64))
batch_size = int(config["data"].get("batch_size", 64))
num_workers = int(config["data"].get("num_workers", 0))
metadata_path = REPO_ROOT / config["data"]["metadata_csv"]
metadata = pd.read_csv(metadata_path)

display(metadata.head())
display(metadata["split"].value_counts().rename_axis("split").to_frame("count"))
display(metadata["is_anomaly"].value_counts().rename_axis("is_anomaly").to_frame("count"))

train_dataset = WaferMapDataset(metadata_path, split="train", image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split="val", image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split="test", image_size=image_size)

pin_memory = device.type == "cuda"
persistent_workers = num_workers > 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

print(f"train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
class WideResNet50_2FeatureExtractor(torch.nn.Module):
    """Pretrained WideResNet50-2 adapted for single-channel wafer maps."""

    def __init__(
        self,
        pretrained: bool = True,
        input_size: int = 224,
        freeze_backbone: bool = True,
        normalize_imagenet: bool = True,
    ) -> None:
        super().__init__()
        weights = Wide_ResNet50_2_Weights.DEFAULT if pretrained else None
        backbone = wide_resnet50_2(weights=weights)

        original_conv = backbone.conv1
        adapted_conv = torch.nn.Conv2d(
            1,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            adapted_conv.weight.copy_(original_conv.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = adapted_conv

        self.stem = torch.nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.layers = torch.nn.Sequential(
            self.layer1,
            self.layer2,
            self.layer3,
            self.layer4,
        )
        self.avgpool = backbone.avgpool
        self.embedding_dim = backbone.fc.in_features
        self.input_size = int(input_size)
        self.normalize_imagenet = bool(normalize_imagenet)
        self.register_buffer("image_mean", torch.tensor([0.4490], dtype=torch.float32).view(1, 1, 1, 1))
        self.register_buffer("image_std", torch.tensor([0.2260], dtype=torch.float32).view(1, 1, 1, 1))

        if freeze_backbone:
            for parameter in self.parameters():
                parameter.requires_grad = False

    def preprocess(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.input_size or x.shape[-2] != self.input_size:
            x = torch.nn.functional.interpolate(
                x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False
            )
        if self.normalize_imagenet:
            x = (x - self.image_mean) / self.image_std
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.preprocess(x)
        x = self.stem(x)
        x = self.layers(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

model = WideResNet50_2FeatureExtractor(
    pretrained=bool(config["model"].get("pretrained", True)),
    input_size=int(config["model"].get("input_size", 224)),
    freeze_backbone=bool(config["model"].get("freeze_backbone", True)),
    normalize_imagenet=bool(config["model"].get("normalize_imagenet", True)),
).to(device)

def collect_embeddings(dataloader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    embeddings = []
    labels = []
    model.eval()

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            for inputs, batch_labels in dataloader:
                inputs = inputs.to(device, non_blocking=True)
                batch_embeddings = model(inputs)
                embeddings.append(batch_embeddings.detach().cpu())
                labels.append(batch_labels.cpu())

    embeddings = torch.cat(embeddings, dim=0).numpy().astype(np.float32)
    labels = torch.cat(labels, dim=0).numpy()
    return embeddings, labels

def l2_center_scores(embeddings: np.ndarray, center: np.ndarray) -> np.ndarray:
    return np.linalg.norm(embeddings - center[None, :], axis=1).astype(np.float32)

model = model.to(device)
model.eval()

print("Resolved device:", device)
print("Model device:", next(model.parameters()).device)
print("Batch size:", train_loader.batch_size)
print("Num workers:", train_loader.num_workers)

In [ ]:
from tqdm.auto import tqdm
import time

print("========== Evaluation Pipeline ==========")

t0 = time.time()

output_dir = REPO_ROOT / config["run"]["output_dir"]
eval_dir = output_dir / "evaluation"
output_dir.mkdir(parents=True, exist_ok=True)
eval_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir)

# -----------------------------
# Collect embeddings
# -----------------------------
print("\n[1/4] Collecting embeddings")

print(" -> Train embeddings")
train_embeddings, train_labels = collect_embeddings(tqdm(train_loader))

print(" -> Validation embeddings")
val_embeddings, val_labels = collect_embeddings(tqdm(val_loader))

print(" -> Test embeddings")
test_embeddings, test_labels = collect_embeddings(tqdm(test_loader))

# -----------------------------
# Compute anomaly scores
# -----------------------------
print("\n[2/4] Computing center and anomaly scores")

train_center = train_embeddings.mean(axis=0).astype(np.float32)

val_scores = l2_center_scores(val_embeddings, train_center)
test_scores = l2_center_scores(test_embeddings, train_center)

val_scores_df = pd.DataFrame({
    "score": val_scores,
    "is_anomaly": val_labels.astype(int)
})

test_scores_df = pd.DataFrame({
    "score": test_scores,
    "is_anomaly": test_labels.astype(int)
})

# -----------------------------
# Threshold selection
# -----------------------------
print("\n[3/4] Selecting threshold")

threshold_quantile = float(config["scoring"].get("threshold_quantile", 0.95))

val_normal_scores = val_scores_df.loc[
    val_scores_df["is_anomaly"] == 0,
    "score"
]

threshold = float(val_normal_scores.quantile(threshold_quantile))

metrics = summarize_threshold_metrics(
    test_labels.astype(int),
    test_scores,
    threshold
)

threshold_sweep_df, best_sweep = sweep_threshold_metrics(
    test_labels.astype(int),
    test_scores
)

# -----------------------------
# Save outputs
# -----------------------------
print("\n[4/4] Saving evaluation artifacts")

np.save(output_dir / "train_center.npy", train_center)

val_scores_df.to_csv(eval_dir / "val_scores.csv", index=False)
test_scores_df.to_csv(eval_dir / "test_scores.csv", index=False)
threshold_sweep_df.to_csv(eval_dir / "threshold_sweep.csv", index=False)

summary = {
    "backbone": "wideresnet50_2",
    "pretrained": bool(config["model"].get("pretrained", True)),
    "freeze_backbone": bool(config["model"].get("freeze_backbone", True)),
    "input_size": int(config["model"].get("input_size", 224)),
    "embedding_dim": int(train_embeddings.shape[1]),
    "train_center_norm": float(np.linalg.norm(train_center)),
    "threshold_quantile": threshold_quantile,
    "threshold": threshold,
    "metrics_at_validation_threshold": metrics,
    "best_threshold_sweep": best_sweep,
}

with (eval_dir / "summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

elapsed = time.time() - t0

print("\nSaved outputs to:", output_dir)
print(f"Total runtime: {elapsed:.2f}s")

summary

In [ ]:
metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "auroc", "value": metrics["auroc"]},
        {"metric": "auprc", "value": metrics["auprc"]},
        {"metric": "threshold", "value": threshold},
    ]
)

display(metrics_df)
display(pd.DataFrame(metrics["confusion_matrix"], index=["true_normal", "true_anomaly"], columns=["pred_normal", "pred_anomaly"]))
print(f"Best sweep threshold: {best_sweep['threshold']:.6f} | precision={best_sweep['precision']:.4f}, recall={best_sweep['recall']:.4f}, f1={best_sweep['f1']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.8, color="#4d908e")
axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[0].set_title("Validation Normal Score Distribution")
axes[0].legend()

axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly")
axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[1].set_title("Test Score Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
plt.axvline(threshold, color="red", linestyle="--", label=f"validation threshold = {threshold:.4f}")
plt.axvline(best_sweep["threshold"], color="green", linestyle=":", label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
plt.title("Threshold Sweep on Test Split")
plt.xlabel("Anomaly-score threshold")
plt.ylabel("Metric value")
plt.legend()
plt.show()

In [ ]:
rng = np.random.default_rng(int(config["run"]["seed"]))
val_normal_idx = np.where(val_labels == 0)[0]
test_anomaly_idx = np.where(test_labels == 1)[0]
test_normal_idx = np.where(test_labels == 0)[0]

sampled_val_normal = rng.choice(val_normal_idx, size=min(PCA_MAX_POINTS_PER_GROUP, len(val_normal_idx)), replace=False)
sampled_test_normal = rng.choice(test_normal_idx, size=min(PCA_MAX_POINTS_PER_GROUP, len(test_normal_idx)), replace=False)
sampled_test_anomaly = rng.choice(test_anomaly_idx, size=min(PCA_MAX_POINTS_PER_GROUP, len(test_anomaly_idx)), replace=False)

pca_embeddings = np.concatenate([
    val_embeddings[sampled_val_normal],
    test_embeddings[sampled_test_normal],
    test_embeddings[sampled_test_anomaly],
], axis=0)
pca_labels = (
    ["val_normal"] * len(sampled_val_normal)
    + ["test_normal"] * len(sampled_test_normal)
    + ["test_anomaly"] * len(sampled_test_anomaly)
)

pca = PCA(n_components=2, random_state=int(config["run"]["seed"]))
pca_points = pca.fit_transform(pca_embeddings)
pca_df = pd.DataFrame({"pc1": pca_points[:, 0], "pc2": pca_points[:, 1], "group": pca_labels})

fig, ax = plt.subplots(figsize=(7, 6))
for group, color in [("val_normal", "#277da1"), ("test_normal", "#90be6d"), ("test_anomaly", "#f94144")]:
    subset = pca_df[pca_df["group"] == group]
    ax.scatter(subset["pc1"], subset["pc2"], s=18, alpha=0.6, label=group, color=color)
ax.set_title("PCA of ResNet18 Embeddings")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
plt.tight_layout()
plt.show()

pca_df.head()

## Failure Analysis

This section attaches the WideResNet50-2 distance scores to the test metadata and highlights the main error patterns under the validation-derived threshold.

In [ ]:
analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
analysis_df["score"] = test_scores_df.reset_index(drop=True)["score"]
analysis_df["predicted_anomaly"] = (analysis_df["score"] > threshold).astype(int)

analysis_df["error_type"] = "tn"
analysis_df.loc[(analysis_df["is_anomaly"] == 0) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "fp"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 0), "error_type"] = "fn"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "tp"

error_summary_df = (
    analysis_df.groupby("error_type")
    .agg(count=("error_type", "size"), mean_score=("score", "mean"))
    .reindex(["tp", "fn", "fp", "tn"])
)

defect_recall_df = (
    analysis_df[analysis_df["is_anomaly"] == 1]
    .groupby("defect_type")
    .agg(count=("defect_type", "size"), detected=("predicted_anomaly", "sum"), mean_score=("score", "mean"))
    .sort_values(["detected", "count"], ascending=[False, False])
)
defect_recall_df["recall"] = defect_recall_df["detected"] / defect_recall_df["count"]

display(error_summary_df)
display(defect_recall_df)
analysis_df.head()